# Sujet 06 — Résolution interactive

Ce notebook illustre l'utilisation de la librairie `consulting_scheduler` sur quelques instances.

**Prérequis** : le package doit être installé en mode éditable (`pip install -e ".[dev]"` à la racine du projet).

In [ ]:
from consulting_scheduler import Problem, WeekTask, solve, solve_schedule

## Exemple 1 — repos / difficile alternés

Quand `h_i` est beaucoup plus élevé que `l_i` pour les semaines paires, la solution optimale consiste à alterner repos et difficile.

In [ ]:
easy = [10, 1, 10, 10]
hard = [5, 50, 5, 50]

result = solve_schedule(easy, hard)
print(result.pretty())

## Exemple 2 — les faciles dominent

Avec des profits difficiles à peine meilleurs que les faciles, la contrainte de repos rend le difficile peu rentable.

In [ ]:
easy = [10, 10, 10, 10]
hard = [5, 12, 5, 12]

result = solve_schedule(easy, hard)
print(result.pretty())

## Exemple 3 — API orientée objet

On peut aussi construire une instance `Problem` et la passer à `solve`.

In [ ]:
pb = Problem.from_sequences(easy=[10, 1, 10, 1, 10], hard=[5, 50, 5, 50, 5])
result = solve(pb)

print(f"Gain optimal : {result.total_gain}")
for i, task in enumerate(result.schedule, start=1):
    print(f"  Semaine {i} : {task.value}")

## Exemple 4 — génération aléatoire et vérification de cohérence

In [ ]:
import random

rng = random.Random(0)
n = 12
easy = [rng.randint(0, 50) for _ in range(n)]
hard = [rng.randint(0, 120) for _ in range(n)]
print(f"easy = {easy}")
print(f"hard = {hard}")

result = solve_schedule(easy, hard)

# Vérification : somme des profits effectifs = gain annoncé
total = sum(
    (easy[i] if t is WeekTask.EASY else hard[i] if t is WeekTask.HARD else 0)
    for i, t in enumerate(result.schedule)
)
assert total == result.total_gain, (total, result.total_gain)

# Vérification : chaque HARD est précédé d'un REST
for i, t in enumerate(result.schedule):
    if t is WeekTask.HARD and i >= 1:
        assert result.schedule[i - 1] is WeekTask.REST

print()
print(result.pretty())